Library of whale calls and songs.
Use this library to validate data manually by gaining information on types of calls and songs, their frequency ranges, their spectrograms and what they sound like. It also shows what the model predicts.

Could maybe add: measured call duration, waveform and frequency range? If that's the case, I will maybe change the layout a bit (if possible), or see if I can do another dropdown menu after using the first one.

In [9]:
# Import of libraries
from IPython.display import display, Markdown
import ipywidgets as widgets
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import io

# Import model (I had an issue so I asked AI to help troubleshoot because I don't have tensorflow)
try:
    from classification import run_classification_model
    MODEL_AVAILABLE = True
except Exception as e:
    print("Model not available:", e)

    MODEL_AVAILABLE = False

    # Dummy function to simulate the classification model's output
    def run_classification_model(wav_bytes, model_name=None):
        return {
            "event": ["Narwhal", "Fin whale", "Humpback whale", "Beluga whale"],
            "probability": [0.25, 0.25, 0.25, 0.25],
            "predicted": [0, 0, 0, 0]
        }


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\tilde\anaconda3\envs\fuaos\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: UNKNOWN ERROR (None): Could not find module '_pywrap_tensorflow_common.dll' (or one of its dependencies). Try using the full path with constructor syntax.
Model not available: Traceback (most recent call last):
  File "c:\Users\tilde\anaconda3\envs\fuaos\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: Det angivne modul blev ikke fundet.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.


In [10]:
# Info about the whales, known spectograms and audio files, and where you can insert your unknown files for compairson 
whale_data = {
    "Narwhal": {
        "call_types": "gonna insert my text here later:)",
        "frequency_range": "gonna insert my text here later:)",
        "typical_duration": "gonna insert my text here later:)",
        "example_files": ["known_samples/narwhal/Narwhal.wav", "known_samples/narwhal/narwhal-voicesofthesea.wav"],
        "unknown_files": ["known_samples/beluga/beluga-voicesofthesea.wav"] #insert the actual unknown file here
    },
    "Fin whale": {
        "call_types": "gonna insert my text here later:)",
        "frequency_range": "gonna insert my text here later:)",
        "typical_duration": "gonna insert my text here later:)",
        "example_files": ["known_samples/fin/fin.wav","known_samples/fin/finWhale.wav"],
        "unknown_files": ["known_samples/beluga/beluga-voicesofthesea.wav"] #insert the actual unknown file here
    },
    "Humpback whale": {
        "call_types": "gonna insert my text here later:)",
        "frequency_range": "gonna insert my text here later:)",
        "typical_duration": "gonna insert my text here later:)",
        "example_files": ["known_samples/humpback/humpback_bubblenetFeeding.wav", "known_samples/humpback/humpback_socialSounds.wav"],
        "unknown_files": ["known_samples/beluga/beluga-voicesofthesea.wav"] #insert the actual unknown file here
    },
    "Beluga whale": {
        "call_types": "gonna insert my text here later:)",
        "frequency_range": "gonna insert my text here later:)",
        "typical_duration": "gonna insert my text here later:)",
        "example_files": ["known_samples/beluga/beluga_clicks.wav", "known_samples/beluga/beluga_socialSounds.wav"],
        "unknown_files": ["known_samples/beluga/beluga-voicesofthesea.wav"] #insert the actual unknown file here
    }
}

In [ ]:
# I just checked if I can use the model, which I can't because of tensorflow. 
# Maybe one of you guys can get the actual model output instead of the dummy model?

print("MODEL AVAILABLE:", MODEL_AVAILABLE) 

MODEL AVAILABLE: False


In [3]:
# The dropdown menu where you select which whale you want to look at
whale_selector = widgets.Dropdown(
    options=list(whale_data.keys()),
    description="Species:"
)

output = widgets.Output()
display(whale_selector,output)

Dropdown(description='Species:', options=('Narwhal', 'Fin whale', 'Humpback whale', 'Beluga whale'), value='Na…

Output()

In [4]:
# The spectrograms and audio players that show up when you select a whale

def plot_spectrogram(y, sr, title="Spectrogram"):
    plt.figure(figsize=(8, 3))

    D = librosa.stft(y)
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)

    plt.show()


def show_audio(file_path):
    y, sr = librosa.load(file_path, sr=None)
    return y, sr


def audio_to_bytes(y, sr):
    import soundfile as sf
    buffer = io.BytesIO()
    sf.write(buffer, y, sr, format='WAV')
    return buffer.getvalue()

In [5]:
# This is a test, but I'm trying to see if I can get our model to do the library as well

def run_model_on_audio(y, sr):
    wav_bytes = audio_to_bytes(y, sr)

    result = run_classification_model(
        wav_bytes,
        model_name="my_classifier"
    )

    return result


def display_model_results(result,selected_whale):
    events = result["event"]
    probs = result["probability"]

    display(Markdown("### Model classification result (0-1)"))

    for e, p in zip(events, probs):
        if e.lower() == selected_whale.lower():
            display(Markdown(f"**{e} → {p:.2f}**"))

In [6]:
# Putting everything together

def show_whale_info(change):
    output.clear_output()

    whale = change["new"]
    data = whale_data[whale]

    with output:
        # The whale info
        display(Markdown(f"### {whale}"))
        display(Markdown(f"**Call types:** {data['call_types']}"))
        display(Markdown(f"**Frequency range:** {data['frequency_range']}"))
        display(Markdown(f"**Typical duration:** {data['typical_duration']}"))

        display(Markdown("---"))
        display(Markdown("Examples"))

        # The spectrograms and audio
        for file_path in data["example_files"]:
            display(Markdown(f"### {file_path}"))

            try:
                y, sr = show_audio(file_path)

                # Spectrogram
                plot_spectrogram(y, sr, title=file_path)

                # Audio
                display(Audio(y, rate=sr))

                # Model
                result = run_model_on_audio(y, sr)
                display_model_results(result,whale)

                display(Markdown("---"))

            except Exception as e:
                print(f"Couldn't load {file_path}: {e}")

        display(Markdown("Unknown files"))        

        for file_path in data["unknown_files"]:
            display(Markdown(f"### {file_path}"))

            try:
                y, sr = show_audio(file_path)

                # Spectrogram
                plot_spectrogram(y, sr, title=file_path)

                # Audio
                display(Audio(y, rate=sr))

                # Model
                result = run_model_on_audio(y, sr)
                display_model_results(result,whale)

                display(Markdown("---"))

            except Exception as e:
                print(f"Couldn't load {file_path}: {e}")  

In [7]:
# Finalizing UI

whale_selector.observe(show_whale_info, names="value")